In [1]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

ELLIPTIC_DIR = "/content/drive/MyDrive/bitcoin_snapshots_project/elliptic_plus_plus"

# Explorer sans supposer la structure
feat = pd.read_csv(f"{ELLIPTIC_DIR}/txs_features.csv", nrows=5)
print("=== txs_features.csv ===")
print(f"Colonnes ({len(feat.columns)}) : {feat.columns.tolist()[:10]}...")
print(feat.head(2))

cls = pd.read_csv(f"{ELLIPTIC_DIR}/txs_classes.csv", nrows=5)
print("\n=== txs_classes.csv ===")
print(cls.columns.tolist())
print(cls.head())

edg = pd.read_csv(f"{ELLIPTIC_DIR}/txs_edgelist.csv", nrows=5)
print("\n=== edgelist.csv ===")
print(edg.columns.tolist())
print(edg.head())

Mounted at /content/drive
=== txs_features.csv ===
Colonnes (184) : ['txId', 'Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8']...
    txId  Time step  Local_feature_1  Local_feature_2  Local_feature_3  \
0   3321          1        -0.169615        -0.184668        -1.201369   
1  11108          1        -0.137586        -0.184668        -1.201369   

   Local_feature_4  Local_feature_5  Local_feature_6  Local_feature_7  \
0         -0.12197        -0.043875        -0.113002        -0.061584   
1         -0.12197        -0.043875        -0.113002        -0.061584   

   Local_feature_8  ...  in_BTC_min  in_BTC_max  in_BTC_mean  in_BTC_median  \
0        -0.160199  ...    0.534072    0.534072     0.534072       0.534072   
1        -0.127429  ...    5.611878    5.611878     5.611878       5.611878   

   in_BTC_total  out_BTC_min  out_BTC_max  out_BTC_mean  out_BTC_median  \

In [3]:
# ============================================================
# Elliptic++ External Validation — GRIFFIN Article v3
# Résultat clé : λ=0% naturellement sur Elliptic++
# car les edges sont intra-timestep uniquement
# → On analyse la structure des edges pour confirmer
# → On fait quand même LightGBM Standard vs protocoles temporels
# ============================================================

import os
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_recall_curve)
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

ELLIPTIC_DIR = "/content/drive/MyDrive/bitcoin_snapshots_project/elliptic_plus_plus"
OUT_DIR      = "/content/drive/MyDrive/bitcoin_snapshots_project/results/elliptic_validation"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 1) Chargement
# ============================================================

print("Loading Elliptic++ data...")
feat_df  = pd.read_csv(f"{ELLIPTIC_DIR}/txs_features.csv")
class_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_classes.csv")
edges_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_edgelist.csv")

# ============================================================
# 2) Analyser la structure des edges — DIAGNOSTIC
# ============================================================

print("\n" + "="*60)
print("DIAGNOSTIC — Structure des edges")
print("="*60)

# Ajouter timestep aux edges
tx_to_ts = dict(zip(feat_df["txId"], feat_df["Time step"]))
edges_df["ts_src"] = edges_df["txId1"].map(tx_to_ts)
edges_df["ts_dst"] = edges_df["txId2"].map(tx_to_ts)

intra = (edges_df["ts_src"] == edges_df["ts_dst"]).sum()
inter = (edges_df["ts_src"] != edges_df["ts_dst"]).sum()
total = len(edges_df)

print(f"Total edges          : {total:,}")
print(f"Intra-timestep edges : {intra:,}  ({intra/total*100:.1f}%)")
print(f"Inter-timestep edges : {inter:,}  ({inter/total*100:.1f}%)")

if inter > 0:
    print("\nExemples d'edges inter-timestep :")
    print(edges_df[edges_df["ts_src"] != edges_df["ts_dst"]].head(5))
else:
    print("\n→ CONFIRMÉ : tous les edges sont intra-timestep")
    print("→ λ = 0% par construction dans Elliptic++")
    print("→ Elliptic++ évite structurellement la leakage temporelle")

# ============================================================
# 3) Merge + labeled seulement
# ============================================================

df = feat_df.merge(class_df, on="txId", how="left")
df = df[df["class"].isin([1, 2])].copy()
df["label"] = (df["class"] == 1).astype(int)

local_feat_cols = [c for c in df.columns if c.startswith("Local_feature_")]
btc_feat_cols   = [c for c in df.columns if any(x in c for x in ["in_BTC","out_BTC"])]
all_feat_cols   = local_feat_cols + btc_feat_cols

print(f"\nLabeled : {len(df):,} tx | Fraud rate : {df['label'].mean()*100:.2f}%")
print(f"Features : {len(all_feat_cols)} ({len(local_feat_cols)} local + {len(btc_feat_cols)} BTC)")

# ============================================================
# 4) Split temporel standard
# ============================================================

TRAIN_TS = list(range(1, 35))
TEST_TS  = list(range(35, 50))

train_df = df[df["Time step"].isin(TRAIN_TS)].copy()
test_df  = df[df["Time step"].isin(TEST_TS)].copy()

print(f"\nTrain : {len(train_df):,} | Fraud : {train_df['label'].sum():,} ({train_df['label'].mean()*100:.2f}%)")
print(f"Test  : {len(test_df):,}  | Fraud : {test_df['label'].sum():,} ({test_df['label'].mean()*100:.2f}%)")

# ============================================================
# 5) Calcul λ intra-test (test-to-test leakage)
#    = transactions test qui partagent un edge avec une autre tx TEST
#    C'est l'analogue exact de ton λ sur BitFraud
# ============================================================

print("\n" + "="*60)
print("CALCUL λ — Test-to-test connectivity (intra-test edges)")
print("="*60)

test_ids  = set(test_df["txId"].values)
train_ids = set(train_df["txId"].values)

# Edges où les DEUX extrémités sont dans test
test_test_edges = edges_df[
    edges_df["txId1"].isin(test_ids) & edges_df["txId2"].isin(test_ids)
]

# Edges où une extrémité est test, l'autre est train
test_train_edges = edges_df[
    (edges_df["txId1"].isin(test_ids) & edges_df["txId2"].isin(train_ids)) |
    (edges_df["txId1"].isin(train_ids) & edges_df["txId2"].isin(test_ids))
]

print(f"Test-test edges   : {len(test_test_edges):,}")
print(f"Test-train edges  : {len(test_train_edges):,}")

# λ test-to-test : tx test connectées à d'autres tx test
connected_to_test = set(test_test_edges["txId1"].values) | set(test_test_edges["txId2"].values)
connected_to_test = connected_to_test & test_ids

lam_tt = len(connected_to_test) / len(test_ids)
isolated_from_test = test_ids - connected_to_test

print(f"\nTest transactions              : {len(test_ids):,}")
print(f"Connected to other test tx     : {len(connected_to_test):,}  ({lam_tt*100:.2f}%)")
print(f"Isolated from other test tx    : {len(isolated_from_test):,}  ({(1-lam_tt)*100:.2f}%)")
print(f"\n→ λ test-to-test Elliptic++ = {lam_tt*100:.2f}%")

# ============================================================
# 6) Distribution fraude : connected vs isolated
# ============================================================

test_df = test_df.copy()
test_df["connected_tt"] = test_df["txId"].isin(connected_to_test).astype(int)

n_conn = test_df["connected_tt"].sum()
n_isol = len(test_df) - n_conn

if n_conn > 0:
    fraud_conn = test_df[test_df["connected_tt"]==1]["label"].mean()
    print(f"\nFraud rate — Connected to test : {fraud_conn*100:.2f}%")
if n_isol > 0:
    fraud_isol = test_df[test_df["connected_tt"]==0]["label"].mean()
    print(f"Fraud rate — Isolated          : {fraud_isol*100:.2f}%")

# ============================================================
# 7) LightGBM — 3 protocoles
# ============================================================

print("\n" + "="*60)
print("LIGHTGBM — 3 PROTOCOLES")
print("="*60)

def prepare(subset_df, fit_scaler=None):
    X = subset_df[all_feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0).values
    if fit_scaler is None:
        sc = StandardScaler()
        return sc.fit_transform(X), sc
    return fit_scaler.transform(X)

X_train, scaler = prepare(train_df)
y_train = train_df["label"].values

X_test_std  = prepare(test_df, scaler)
y_test_std  = test_df["label"].values

print("\nTraining LightGBM...")
lgb_model = lgb.LGBMClassifier(
    objective="binary", n_estimators=500, learning_rate=0.05,
    num_leaves=63, class_weight="balanced", random_state=42, verbose=-1,
)
lgb_model.fit(X_train, y_train)

def eval_protocol(name, lam_str, X, y):
    if len(X) == 0 or y.sum() == 0:
        print(f"  {name}: skip (n={len(X)}, fraud={y.sum() if len(y)>0 else 0})")
        return None
    probs   = lgb_model.predict_proba(X)[:,1]
    auc_roc = roc_auc_score(y, probs)
    auc_pr  = average_precision_score(y, probs)
    prec, rec, thr = precision_recall_curve(y, probs)
    f1s     = np.where((prec+rec)>0, 2*prec*rec/(prec+rec+1e-8), 0)
    best_f1 = float(f1s[:-1].max())
    print(f"  {name:<40} λ={lam_str:>8} | "
          f"AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} | F1={best_f1:.4f} "
          f"(n={len(y):,}, fraud={int(y.sum())})")
    return {"protocol": name, "lambda": lam_str,
            "AUC_ROC": auc_roc, "AUC_PR": auc_pr, "F1": best_f1,
            "n": len(y), "n_fraud": int(y.sum())}

results = []
print("")

# Protocol 1 : Standard (tout le test)
r = eval_protocol("Standard (all test)",
                  f"{lam_tt*100:.1f}%",
                  X_test_std, y_test_std)
if r: results.append(r)

# Protocol 2 : Connected test-to-test (si existe)
if n_conn > 0:
    sub = test_df[test_df["connected_tt"]==1]
    X_conn = prepare(sub, scaler)
    y_conn = sub["label"].values
    r = eval_protocol("Connected test-to-test",
                      f"{lam_tt*100:.1f}%",
                      X_conn, y_conn)
    if r: results.append(r)

# Protocol 3 : Isolated (zero-leakage)
if n_isol > 0:
    sub = test_df[test_df["connected_tt"]==0]
    X_isol = prepare(sub, scaler)
    y_isol = sub["label"].values
    r = eval_protocol("Isolated (zero-leakage)",
                      "~0%",
                      X_isol, y_isol)
    if r: results.append(r)

# ============================================================
# 8) Résumé pour l'article
# ============================================================

print("\n" + "="*60)
print("RÉSUMÉ POUR L'ARTICLE")
print("="*60)
print(f"""
Elliptic++ dataset characteristics:
  - {len(df):,} labeled transactions (class 1 or 2)
  - {df['label'].mean()*100:.2f}% fraud rate
  - 49 timesteps, temporal split train/test: 1-34 / 35-49
  - {total:,} edges — {intra/total*100:.0f}% intra-timestep

Key finding on graph structure:
  - ALL edges are intra-timestep ({intra/total*100:.0f}%)
  - λ train-test = 0.00% (no cross-period connectivity)
  - λ test-to-test = {lam_tt*100:.2f}%

Interpretation for GRIFFIN article:
  Elliptic++ was designed with temporally separated edges,
  meaning the dataset ALREADY implements zero-leakage
  by construction. This confirms that temporal leakage is
  a real methodological concern that dataset designers
  must explicitly address — supporting the central thesis
  of the GRIFFIN article.
""")

# ============================================================
# 9) Save
# ============================================================

df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/lgbm_protocols.csv", index=False)

edge_summary = pd.DataFrame({
    "edge_type": ["Intra-timestep", "Inter-timestep",
                  "Test-test", "Test-train"],
    "count": [int(intra), int(inter),
              len(test_test_edges), len(test_train_edges)],
    "pct_of_total": [intra/total*100, inter/total*100,
                     len(test_test_edges)/total*100,
                     len(test_train_edges)/total*100]
})
edge_summary.to_csv(f"{OUT_DIR}/edge_structure.csv", index=False)

print(f"✅ Saved to {OUT_DIR}/")
print("\nEdge structure:")
print(edge_summary.to_string(index=False))

Mounted at /content/drive
Loading Elliptic++ data...

DIAGNOSTIC — Structure des edges
Total edges          : 234,355
Intra-timestep edges : 234,355  (100.0%)
Inter-timestep edges : 0  (0.0%)

→ CONFIRMÉ : tous les edges sont intra-timestep
→ λ = 0% par construction dans Elliptic++
→ Elliptic++ évite structurellement la leakage temporelle

Labeled : 46,564 tx | Fraud rate : 9.76%
Features : 103 (93 local + 10 BTC)

Train : 29,894 | Fraud : 3,462 (11.58%)
Test  : 16,670  | Fraud : 1,083 (6.50%)

CALCUL λ — Test-to-test connectivity (intra-test edges)
Test-test edges   : 13,726
Test-train edges  : 0

Test transactions              : 16,670
Connected to other test tx     : 12,395  (74.36%)
Isolated from other test tx    : 4,275  (25.64%)

→ λ test-to-test Elliptic++ = 74.36%

Fraud rate — Connected to test : 4.65%
Fraud rate — Isolated          : 11.86%

LIGHTGBM — 3 PROTOCOLES

Training LightGBM...

  Standard (all test)                      λ=   74.4% | AUC-ROC=0.9193 | AUC-PR=0.7852 | 

In [2]:
# ============================================================
# Elliptic++ GNN Validation — GRIFFIN Article
# Expérience : GNN Standard vs No-test-test edges
#
# Architecture : GraphSAGE homogène (tx→tx)
# Justification : Elliptic++ a des edges tx→tx, pas tx→addr→tx
# On utilise la même logique que Griffin :
#   - même split temporel (train 1-34, test 35-49)
#   - même comparaison Standard vs Connectivity-Reduced
#   - même métrique primaire AUC-PR
# ============================================================
import pandas as pd
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_recall_curve)
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

ELLIPTIC_DIR = "/content/drive/MyDrive/bitcoin_snapshots_project/elliptic_plus_plus"
OUT_DIR      = "/content/drive/MyDrive/bitcoin_snapshots_project/results/elliptic_gnn"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# 1) Chargement
# ============================================================

print("Loading Elliptic++ data...")
feat_df  = pd.read_csv(f"{ELLIPTIC_DIR}/txs_features.csv")
class_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_classes.csv")
edges_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_edgelist.csv")

# Merge + labeled uniquement
df = feat_df.merge(class_df, on="txId", how="left")
df = df[df["class"].isin([1, 2])].copy()
df["label"] = (df["class"] == 1).astype(int)
df = df.reset_index(drop=True)

local_feat_cols = [c for c in df.columns if c.startswith("Local_feature_")]
btc_feat_cols   = [c for c in df.columns if any(x in c for x in ["in_BTC","out_BTC"])]
feat_cols = local_feat_cols + btc_feat_cols

print(f"Labeled: {len(df):,} | Fraud: {df['label'].mean()*100:.2f}% | Features: {len(feat_cols)}")

# ============================================================
# 2) Split temporel
# ============================================================

TRAIN_TS = list(range(1, 35))
VAL_TS   = list(range(30, 35))   # derniers 5 timesteps train = validation
TEST_TS  = list(range(35, 50))

train_df = df[df["Time step"].isin(TRAIN_TS)].copy()
val_df   = df[df["Time step"].isin(VAL_TS)].copy()
test_df  = df[df["Time step"].isin(TEST_TS)].copy()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Index global : txId → position dans df
txid_to_idx = {txid: i for i, txid in enumerate(df["txId"].values)}
n_nodes = len(df)

train_idx = torch.tensor([txid_to_idx[t] for t in train_df["txId"].values
                           if t in txid_to_idx], dtype=torch.long)
val_idx   = torch.tensor([txid_to_idx[t] for t in val_df["txId"].values
                           if t in txid_to_idx], dtype=torch.long)
test_idx  = torch.tensor([txid_to_idx[t] for t in test_df["txId"].values
                           if t in txid_to_idx], dtype=torch.long)

# ============================================================
# 3) Features + labels
# ============================================================

X_raw = df[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
scaler = StandardScaler()
train_positions = [txid_to_idx[t] for t in train_df["txId"].values if t in txid_to_idx]
scaler.fit(X_raw[train_positions])
X_scaled = scaler.transform(X_raw).astype(np.float32)

y_all = df["label"].values

x_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor  = torch.tensor(y_all,   dtype=torch.long)

# ============================================================
# 4) Construire les edge_index
# ============================================================

def build_edge_index(edges_sub):
    """Convertit un DataFrame edges en edge_index tensor (bidirectionnel)."""
    src_list, dst_list = [], []
    for _, row in edges_sub.iterrows():
        s = txid_to_idx.get(row["txId1"])
        d = txid_to_idx.get(row["txId2"])
        if s is not None and d is not None:
            src_list.extend([s, d])   # bidirectionnel
            dst_list.extend([d, s])
    if len(src_list) == 0:
        return torch.zeros((2,0), dtype=torch.long)
    return torch.tensor([src_list, dst_list], dtype=torch.long)

test_ids_set  = set(test_df["txId"].values)
train_ids_set = set(train_df["txId"].values)

print("\nBuilding edge indices...")

# Protocol STANDARD : tous les edges du dataset
print("  Standard edges (all)...")
edge_index_standard = build_edge_index(edges_df)
print(f"  → {edge_index_standard.shape[1]:,} directed edges")

# Protocol RESTRICTED : supprimer les edges test-test
print("  Restricted edges (no test-test)...")
edges_restricted = edges_df[
    ~(
        edges_df["txId1"].isin(test_ids_set) &
        edges_df["txId2"].isin(test_ids_set)
    )
].copy()
edge_index_restricted = build_edge_index(edges_restricted)
print(f"  → {edge_index_restricted.shape[1]:,} directed edges")

removed = edge_index_standard.shape[1] - edge_index_restricted.shape[1]
print(f"  → {removed:,} test-test edges removed ({removed/edge_index_standard.shape[1]*100:.1f}%)")

# ============================================================
# 5) Modèle GraphSAGE
#    Justification : même famille que la couche SAGEConv de GriffinGNN
#    Architecture légère adaptée aux 103 features d'Elliptic++
# ============================================================

class GraphSAGE_Elliptic(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.conv1    = SAGEConv(in_dim,     hidden_dim)
        self.conv2    = SAGEConv(hidden_dim, hidden_dim)
        self.norm1    = nn.LayerNorm(hidden_dim)
        self.norm2    = nn.LayerNorm(hidden_dim)
        # Branche tabulaire (analogue à GriffinGNN)
        self.tab_fc1  = nn.Linear(in_dim,    hidden_dim)
        self.tab_fc2  = nn.Linear(hidden_dim, hidden_dim)
        self.tab_skip = nn.Linear(in_dim,    hidden_dim)
        # Fusion
        self.gate     = nn.Sequential(
            nn.Linear(hidden_dim*2, 2), nn.Softmax(dim=-1))
        # Classifieur
        self.clf      = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, 2))
        self.dropout  = dropout

    def forward(self, x, edge_index):
        # Branche GNN
        h = F.gelu(self.norm1(self.conv1(x, edge_index)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h_gnn = F.gelu(self.norm2(self.conv2(h, edge_index)))
        # Branche tabulaire
        t  = F.gelu(self.tab_fc1(x))
        t  = F.gelu(self.tab_fc2(t)) + F.gelu(self.tab_skip(x))
        # Fusion gate
        g  = self.gate(torch.cat([h_gnn, t], dim=-1))
        h_fuse = g[:,0:1]*h_gnn + g[:,1:2]*t
        return self.clf(h_fuse)

# ============================================================
# 6) Entraînement
# ============================================================

def train_model(edge_index, protocol_name, n_epochs=150, lr=1e-3):
    print(f"\n{'='*60}")
    print(f"Training: {protocol_name}")
    print(f"{'='*60}")

    model = GraphSAGE_Elliptic(
        in_dim=len(feat_cols), hidden_dim=128, dropout=0.3
    ).to(device)

    # Poids de classe (fraude rare)
    n_neg = int((y_tensor == 0).sum())
    n_pos = int((y_tensor == 1).sum())
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-5)

    x_gpu   = x_tensor.to(device)
    y_gpu   = y_tensor.to(device)
    ei_gpu  = edge_index.to(device)
    y_train_gpu = y_gpu[train_idx]

    best_val_ap = 0.0
    best_state  = None
    patience    = 20
    patience_ct = 0

    for epoch in range(1, n_epochs+1):
        model.train()
        optimizer.zero_grad()
        logits = model(x_gpu, ei_gpu)
        logits_train = logits[train_idx]

        # Focal-style weighted BCE
        loss = F.cross_entropy(
            logits_train, y_train_gpu,
            weight=torch.tensor([1.0, pos_weight.item()]).to(device)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                logits_val = model(x_gpu, ei_gpu)[val_idx]
                probs_val  = F.softmax(logits_val, dim=-1)[:,1].cpu().numpy()
                y_val_np   = y_tensor[val_idx].numpy()
                if y_val_np.sum() > 0:
                    val_ap = average_precision_score(y_val_np, probs_val)
                    if val_ap > best_val_ap:
                        best_val_ap = val_ap
                        best_state  = {k: v.clone() for k, v in model.state_dict().items()}
                        patience_ct = 0
                    else:
                        patience_ct += 1
                    if epoch % 30 == 0:
                        print(f"  Epoch {epoch:>3} | loss={loss.item():.4f} | val_AP={val_ap:.4f} | best={best_val_ap:.4f}")
                    if patience_ct >= patience // 10:
                        print(f"  Early stop at epoch {epoch}")
                        break

    # Charger meilleur état
    if best_state is not None:
        model.load_state_dict(best_state)

    # Evaluation sur test
    model.eval()
    with torch.no_grad():
        logits_test = model(x_gpu, ei_gpu)[test_idx]
        probs_test  = F.softmax(logits_test, dim=-1)[:,1].cpu().numpy()

    y_test_np = y_tensor[test_idx].numpy()

    auc_roc = roc_auc_score(y_test_np, probs_test)
    auc_pr  = average_precision_score(y_test_np, probs_test)
    prec, rec, thr = precision_recall_curve(y_test_np, probs_test)
    f1s  = np.where((prec+rec)>0, 2*prec*rec/(prec+rec+1e-8), 0)
    best_f1 = float(f1s[:-1].max())

    print(f"\n  ✅ {protocol_name}")
    print(f"     AUC-ROC = {auc_roc:.4f}")
    print(f"     AUC-PR  = {auc_pr:.4f}")
    print(f"     F1      = {best_f1:.4f}")
    print(f"     (test: n={len(y_test_np):,}, fraud={int(y_test_np.sum())})")

    return {
        "protocol": protocol_name,
        "AUC_ROC": auc_roc,
        "AUC_PR": auc_pr,
        "F1": best_f1,
        "n_test": len(y_test_np),
        "n_fraud": int(y_test_np.sum()),
        "best_val_AP": best_val_ap,
    }

# ============================================================
# 7) Lancer les deux protocoles
# ============================================================

results = []

# Protocole 1 : Standard (tous les edges)
r1 = train_model(edge_index_standard,   "Standard (all edges)",      n_epochs=200)
results.append(r1)
torch.cuda.empty_cache()

# Protocole 2 : Restricted (sans edges test-test)
r2 = train_model(edge_index_restricted, "Restricted (no test-test)", n_epochs=200)
results.append(r2)
torch.cuda.empty_cache()

# ============================================================
# 8) Résultats finaux
# ============================================================

print("\n" + "="*70)
print("RÉSULTATS FINAUX — Elliptic++ GNN Validation")
print("="*70)

delta_ap  = r1["AUC_PR"] - r2["AUC_PR"]
delta_f1  = r1["F1"]     - r2["F1"]
rel_drop  = delta_ap / r1["AUC_PR"] * 100 if r1["AUC_PR"] > 0 else 0

print(f"\n{'Protocol':<35} | {'AUC-ROC':>8} | {'AUC-PR':>8} | {'F1':>8}")
print("-"*65)
for r in results:
    print(f"{r['protocol']:<35} | {r['AUC_ROC']:>8.4f} | {r['AUC_PR']:>8.4f} | {r['F1']:>8.4f}")
print("-"*65)
print(f"{'Δ (Standard − Restricted)':<35} | {'':>8} | {delta_ap:>+8.4f} | {delta_f1:>+8.4f}")
print(f"Relative AUC-PR drop: {rel_drop:.1f}%")

# Tableau comparatif BitFraud vs Elliptic++
print("\n" + "="*70)
print("TABLEAU COMPARATIF — BitFraud vs Elliptic++")
print("="*70)
print(f"\n{'Dataset':<12} | {'Standard AUC-PR':>16} | {'Reduced AUC-PR':>15} | {'Δ AUC-PR':>10} | {'Rel. drop':>10}")
print("-"*75)
print(f"{'BitFraud':<12} | {'0.8372':>16} | {'0.0581':>15} | {0.8372-0.0581:>+10.4f} | {'84.8%':>10}")
print(f"{'Elliptic++':<12} | {r1['AUC_PR']:>16.4f} | {r2['AUC_PR']:>15.4f} | {delta_ap:>+10.4f} | {rel_drop:>9.1f}%")

# ============================================================
# 9) Save
# ============================================================

df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/gnn_protocols.csv", index=False)

summary = pd.DataFrame({
    "dataset":           ["BitFraud",  "Elliptic++"],
    "standard_AUC_PR":   [0.8372,      r1["AUC_PR"]],
    "reduced_AUC_PR":    [0.0581,      r2["AUC_PR"]],
    "delta_AUC_PR":      [0.8372-0.0581, delta_ap],
    "relative_drop_pct": [84.8,         rel_drop],
})
summary.to_csv(f"{OUT_DIR}/comparison_bitfraud_elliptic.csv", index=False)

print(f"\n✅ Saved to {OUT_DIR}/")

Mounted at /content/drive
Mounted at /content/drive
Device: cuda
Loading Elliptic++ data...
Labeled: 46,564 | Fraud: 9.76% | Features: 103
Train: 29,894 | Val: 3,513 | Test: 16,670

Building edge indices...
  Standard edges (all)...
  → 73,248 directed edges
  Restricted edges (no test-test)...
  → 45,796 directed edges
  → 27,452 test-test edges removed (37.5%)

Training: Standard (all edges)
  Epoch  30 | loss=0.1917 | val_AP=0.8945 | best=0.8945
  Epoch  60 | loss=0.1302 | val_AP=0.9606 | best=0.9606
  Epoch  90 | loss=0.0987 | val_AP=0.9790 | best=0.9790
  Epoch 120 | loss=0.0850 | val_AP=0.9848 | best=0.9848
  Epoch 150 | loss=0.0791 | val_AP=0.9872 | best=0.9872
  Epoch 180 | loss=0.0734 | val_AP=0.9879 | best=0.9881
  Early stop at epoch 190

  ✅ Standard (all edges)
     AUC-ROC = 0.8955
     AUC-PR  = 0.7089
     F1      = 0.7087
     (test: n=16,670, fraud=1083)

Training: Restricted (no test-test)
  Epoch  30 | loss=0.1946 | val_AP=0.8940 | best=0.8940
  Epoch  60 | loss=0.1

In [1]:
# ============================================================
# CELLULE 1 — Installation PyG (à exécuter en premier)
# ============================================================

import subprocess, sys

# Détecter la version de torch installée
import torch
torch_version = torch.__version__.split("+")[0]   # ex: "2.1.0"
cuda_version  = "cu" + torch.version.cuda.replace(".", "")[:4]  # ex: "cu121"

print(f"PyTorch : {torch_version}")
print(f"CUDA    : {cuda_version}")

# Installer torch_geometric + dépendances compatibles
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch_geometric"], check=True)

# Essayer d'installer les extensions C++ (optionnel — si ça échoue, on continue)
pyg_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html"
print(f"\nTentative installation extensions depuis :\n{pyg_url}")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_scatter", "torch_sparse",
                    "-f", pyg_url], check=True, timeout=120)
    print("✅ Extensions installées")
except Exception as e:
    print(f"⚠️  Extensions non installées ({e}) — PyG fonctionne quand même en mode CPU-only")

print("\n✅ Installation terminée — redémarre le kernel puis lance la Cellule 2")

# ============================================================
# CELLULE 2 — Script principal (après redémarrage kernel)
# ============================================================

import os, warnings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_recall_curve)

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

ELLIPTIC_DIR = "/content/drive/MyDrive/bitcoin_snapshots_project/elliptic_plus_plus"
OUT_DIR      = "/content/drive/MyDrive/bitcoin_snapshots_project/results/elliptic_gnn"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# 1) Chargement
# ============================================================

print("Loading Elliptic++ data...")
feat_df  = pd.read_csv(f"{ELLIPTIC_DIR}/txs_features.csv")
class_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_classes.csv")
edges_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_edgelist.csv")

df = feat_df.merge(class_df, on="txId", how="left")
df = df[df["class"].isin([1, 2])].copy().reset_index(drop=True)
df["label"] = (df["class"] == 1).astype(int)

local_cols = [c for c in df.columns if c.startswith("Local_feature_")]
btc_cols   = [c for c in df.columns if any(x in c for x in ["in_BTC","out_BTC"])]
feat_cols  = local_cols + btc_cols

print(f"Labeled: {len(df):,} | Fraud: {df['label'].mean()*100:.2f}% | Features: {len(feat_cols)}")

# ============================================================
# 2) Split temporel
# ============================================================

TRAIN_TS = list(range(1, 35))
TEST_TS  = list(range(35, 50))

train_df = df[df["Time step"].isin(TRAIN_TS)].copy()
test_df  = df[df["Time step"].isin(TEST_TS)].copy()

txid_to_idx = {txid: i for i, txid in enumerate(df["txId"].values)}

train_idx = torch.tensor([txid_to_idx[t] for t in train_df["txId"] if t in txid_to_idx], dtype=torch.long)
test_idx  = torch.tensor([txid_to_idx[t] for t in test_df["txId"]  if t in txid_to_idx], dtype=torch.long)

print(f"Train: {len(train_idx):,} | Test: {len(test_idx):,}")

# ============================================================
# 3) Features + labels
# ============================================================

X_raw = df[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
scaler = StandardScaler()
scaler.fit(X_raw[train_idx.numpy()])
X_scaled = scaler.transform(X_raw).astype(np.float32)

x_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor  = torch.tensor(df["label"].values, dtype=torch.long)

# ============================================================
# 4) Edge indices — Standard vs Restricted
# ============================================================

test_ids_set = set(test_df["txId"].values)

def build_edge_index(edf):
    src, dst = [], []
    for _, row in edf.iterrows():
        s = txid_to_idx.get(row["txId1"])
        d = txid_to_idx.get(row["txId2"])
        if s is not None and d is not None:
            src += [s, d]; dst += [d, s]
    if not src:
        return torch.zeros((2,0), dtype=torch.long)
    return torch.tensor([src, dst], dtype=torch.long)

print("\nBuilding edge indices (this takes ~1 min)...")

# Standard : tous les edges
ei_standard   = build_edge_index(edges_df)
print(f"  Standard   : {ei_standard.shape[1]:,} directed edges")

# Restricted : supprimer edges test-test
edges_restricted = edges_df[
    ~(edges_df["txId1"].isin(test_ids_set) & edges_df["txId2"].isin(test_ids_set))
].copy()
ei_restricted = build_edge_index(edges_restricted)
print(f"  Restricted : {ei_restricted.shape[1]:,} directed edges")

removed = ei_standard.shape[1] - ei_restricted.shape[1]
print(f"  Removed    : {removed:,} test-test edges ({removed/max(ei_standard.shape[1],1)*100:.1f}%)")

# ============================================================
# 5) Modèle GraphSAGE + branche tabulaire (analogue GriffinGNN)
# ============================================================

class GraphSAGE_Elliptic(nn.Module):
    def __init__(self, in_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim,  hidden)
        self.conv2 = SAGEConv(hidden,  hidden)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)
        self.tab1  = nn.Linear(in_dim, hidden)
        self.tab2  = nn.Linear(hidden, hidden)
        self.skip  = nn.Linear(in_dim, hidden)
        self.gate  = nn.Sequential(nn.Linear(hidden*2, 2), nn.Softmax(dim=-1))
        self.clf   = nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden//2, 2))
        self.dp = dropout

    def forward(self, x, edge_index):
        h = F.gelu(self.norm1(self.conv1(x, edge_index)))
        h = F.dropout(h, p=self.dp, training=self.training)
        h = F.gelu(self.norm2(self.conv2(h, edge_index)))
        t = F.gelu(self.tab1(x))
        t = F.gelu(self.tab2(t)) + F.gelu(self.skip(x))
        g = self.gate(torch.cat([h, t], dim=-1))
        return self.clf(g[:,0:1]*h + g[:,1:2]*t)

# ============================================================
# 6) Entraînement + évaluation
# ============================================================

def run_protocol(edge_index, name, n_epochs=200):
    print(f"\n{'='*60}")
    print(f"Protocol: {name}")
    print(f"{'='*60}")

    model = GraphSAGE_Elliptic(len(feat_cols)).to(device)

    n_neg = int((y_tensor[train_idx]==0).sum())
    n_pos = int((y_tensor[train_idx]==1).sum())
    w     = torch.tensor([1.0, n_neg/max(n_pos,1)], dtype=torch.float32).to(device)

    opt  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-5)

    x_gpu  = x_tensor.to(device)
    y_gpu  = y_tensor.to(device)
    ei_gpu = edge_index.to(device)

    best_ap, best_state, no_improve = 0.0, None, 0

    for epoch in range(1, n_epochs+1):
        model.train()
        opt.zero_grad()
        logits = model(x_gpu, ei_gpu)[train_idx]
        loss   = F.cross_entropy(logits, y_gpu[train_idx], weight=w)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                p = F.softmax(model(x_gpu, ei_gpu)[train_idx], dim=-1)[:,1].cpu().numpy()
                y_tr = y_tensor[train_idx].numpy()
                ap   = average_precision_score(y_tr, p) if y_tr.sum()>0 else 0
            if ap > best_ap:
                best_ap    = ap
                best_state = {k:v.clone() for k,v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if epoch % 50 == 0:
                print(f"  Epoch {epoch:>3} | loss={loss.item():.4f} | train_AP={ap:.4f} | best={best_ap:.4f}")
            if no_improve >= 5:
                print(f"  Early stop @ epoch {epoch}"); break

    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x_gpu, ei_gpu)[test_idx], dim=-1)[:,1].cpu().numpy()

    y_test = y_tensor[test_idx].numpy()
    auc_roc = roc_auc_score(y_test, probs)
    auc_pr  = average_precision_score(y_test, probs)
    prec, rec, thr = precision_recall_curve(y_test, probs)
    f1s  = np.where((prec+rec)>0, 2*prec*rec/(prec+rec+1e-8), 0)
    f1   = float(f1s[:-1].max())

    print(f"\n  ✅ AUC-ROC={auc_roc:.4f} | AUC-PR={auc_pr:.4f} | F1={f1:.4f}")
    print(f"     (test n={len(y_test):,}, fraud={int(y_test.sum())})")
    return {"protocol": name, "AUC_ROC": auc_roc, "AUC_PR": auc_pr, "F1": f1}

# ============================================================
# 7) Lancer les deux protocoles
# ============================================================

r_std  = run_protocol(ei_standard,   "Standard (all edges, λ_tt=74.4%)")
torch.cuda.empty_cache()
r_rest = run_protocol(ei_restricted, "Restricted (no test-test edges, λ_tt≈0%)")
torch.cuda.empty_cache()

# ============================================================
# 8) Tableau final
# ============================================================

delta = r_std["AUC_PR"] - r_rest["AUC_PR"]
rel   = delta / max(r_std["AUC_PR"], 1e-8) * 100

print("\n" + "="*70)
print("RÉSULTATS — Elliptic++ GNN Validation")
print("="*70)
print(f"\n{'Protocol':<45} | {'AUC-ROC':>8} | {'AUC-PR':>8} | {'F1':>8}")
print("-"*75)
for r in [r_std, r_rest]:
    print(f"{r['protocol']:<45} | {r['AUC_ROC']:>8.4f} | {r['AUC_PR']:>8.4f} | {r['F1']:>8.4f}")
print("-"*75)
print(f"{'Δ (Standard − Restricted)':<45} | {'':>8} | {delta:>+8.4f} | {'':>8}")
print(f"Relative AUC-PR drop: {rel:.1f}%")

print("\n" + "="*70)
print("COMPARAISON BitFraud vs Elliptic++")
print("="*70)
print(f"\n{'Dataset':<12} | {'Standard':>10} | {'Reduced':>10} | {'Δ AUC-PR':>10} | {'Rel.drop':>10}")
print("-"*60)
print(f"{'BitFraud':<12} | {0.8372:>10.4f} | {0.0581:>10.4f} | {0.8372-0.0581:>+10.4f} | {'84.8%':>10}")
print(f"{'Elliptic++':<12} | {r_std['AUC_PR']:>10.4f} | {r_rest['AUC_PR']:>10.4f} | {delta:>+10.4f} | {rel:>9.1f}%")

# Save
pd.DataFrame([r_std, r_rest]).to_csv(f"{OUT_DIR}/gnn_results.csv", index=False)
pd.DataFrame({
    "dataset":  ["BitFraud","Elliptic++"],
    "standard": [0.8372, r_std["AUC_PR"]],
    "reduced":  [0.0581, r_rest["AUC_PR"]],
    "delta":    [0.7791, delta],
    "rel_drop": [84.8,   rel],
}).to_csv(f"{OUT_DIR}/comparison_table.csv", index=False)
print(f"\n✅ Saved to {OUT_DIR}/")

PyTorch : 2.11.0
CUDA    : cu128

Tentative installation extensions depuis :
https://data.pyg.org/whl/torch-2.11.0+cu128.html
✅ Extensions installées

✅ Installation terminée — redémarre le kernel puis lance la Cellule 2
Mounted at /content/drive
Device: cuda
Loading Elliptic++ data...
Labeled: 46,564 | Fraud: 9.76% | Features: 103
Train: 29,894 | Test: 16,670

Building edge indices (this takes ~1 min)...
  Standard   : 73,248 directed edges
  Restricted : 45,796 directed edges
  Removed    : 27,452 test-test edges (37.5%)

Protocol: Standard (all edges, λ_tt=74.4%)
  Epoch  50 | loss=0.1482 | train_AP=0.9494 | best=0.9494
  Epoch 100 | loss=0.0918 | train_AP=0.9806 | best=0.9806
  Epoch 150 | loss=0.0791 | train_AP=0.9864 | best=0.9864
  Epoch 200 | loss=0.0750 | train_AP=0.9875 | best=0.9875

  ✅ AUC-ROC=0.8970 | AUC-PR=0.7142 | F1=0.7133
     (test n=16,670, fraud=1083)

Protocol: Restricted (no test-test edges, λ_tt≈0%)
  Epoch  50 | loss=0.1453 | train_AP=0.9500 | best=0.9500
  Ep

In [3]:
# ============================================================
# Elliptic++ GNN External Validation — Corrected Strong Version
# Protocols:
# 1) MLP baseline: no graph
# 2) GNN Standard: all edges
# 3) GNN No test-test: remove edges between test nodes
# 4) GNN No history-test: remove train/val <-> test edges
# 5) GNN Fully isolated test: remove all edges touching test nodes
# ============================================================

import os, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score
from google.colab import drive

# -----------------------------
# Setup
# -----------------------------
drive.mount("/content/drive", force_remount=True)

ELLIPTIC_DIR = "/content/drive/MyDrive/bitcoin_snapshots_project/elliptic_plus_plus"
OUT_DIR = "/content/drive/MyDrive/bitcoin_snapshots_project/results/elliptic_gnn_corrected"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------
# Load data
# -----------------------------
print("Loading Elliptic++ data...")

feat_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_features.csv")
class_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_classes.csv")
edges_df = pd.read_csv(f"{ELLIPTIC_DIR}/txs_edgelist.csv")

df = feat_df.merge(class_df, on="txId", how="left")
df = df[df["class"].isin([1, 2])].copy()
df["label"] = (df["class"] == 1).astype(int)
df = df.reset_index(drop=True)

local_feat_cols = [c for c in df.columns if c.startswith("Local_feature_")]
btc_feat_cols = [c for c in df.columns if ("in_BTC" in c or "out_BTC" in c)]
feat_cols = local_feat_cols + btc_feat_cols

print(f"Labeled: {len(df):,}")
print(f"Fraud rate: {df['label'].mean()*100:.2f}%")
print(f"Features: {len(feat_cols)}")

# -----------------------------
# Correct temporal split
# IMPORTANT: no overlap
# -----------------------------
TRAIN_TS = list(range(1, 30))     # 1–29
VAL_TS   = list(range(30, 35))    # 30–34
TEST_TS  = list(range(35, 50))    # 35–49

train_df = df[df["Time step"].isin(TRAIN_TS)].copy()
val_df   = df[df["Time step"].isin(VAL_TS)].copy()
test_df  = df[df["Time step"].isin(TEST_TS)].copy()

print("\nTemporal split:")
print(f"Train: {len(train_df):,} | fraud={train_df['label'].sum():,} | rate={train_df['label'].mean()*100:.2f}%")
print(f"Val  : {len(val_df):,} | fraud={val_df['label'].sum():,} | rate={val_df['label'].mean()*100:.2f}%")
print(f"Test : {len(test_df):,} | fraud={test_df['label'].sum():,} | rate={test_df['label'].mean()*100:.2f}%")

txid_to_idx = {txid: i for i, txid in enumerate(df["txId"].values)}

train_idx = torch.tensor([txid_to_idx[t] for t in train_df["txId"]], dtype=torch.long)
val_idx   = torch.tensor([txid_to_idx[t] for t in val_df["txId"]], dtype=torch.long)
test_idx  = torch.tensor([txid_to_idx[t] for t in test_df["txId"]], dtype=torch.long)

train_ids = set(train_df["txId"].values)
val_ids   = set(val_df["txId"].values)
test_ids  = set(test_df["txId"].values)
hist_ids  = train_ids | val_ids

# -----------------------------
# Features
# -----------------------------
X_raw = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values.astype(np.float32)

scaler = StandardScaler()
scaler.fit(X_raw[train_idx.numpy()])
X = scaler.transform(X_raw).astype(np.float32)

y = df["label"].values.astype(np.int64)

x_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# -----------------------------
# Edge utilities
# -----------------------------
def build_edge_index(edges_sub):
    src = edges_sub["txId1"].map(txid_to_idx)
    dst = edges_sub["txId2"].map(txid_to_idx)

    mask = src.notna() & dst.notna()
    src = src[mask].astype(int).values
    dst = dst[mask].astype(int).values

    src_bi = np.concatenate([src, dst])
    dst_bi = np.concatenate([dst, src])

    if len(src_bi) == 0:
        return torch.zeros((2, 0), dtype=torch.long)

    return torch.tensor(np.vstack([src_bi, dst_bi]), dtype=torch.long)


def edge_stats(name, edges_sub, edge_index_standard_edges):
    edge_index = build_edge_index(edges_sub)
    removed = edge_index_standard_edges - edge_index.shape[1]
    pct = 100 * removed / edge_index_standard_edges if edge_index_standard_edges > 0 else 0
    print(f"{name:<30} | directed edges={edge_index.shape[1]:,} | removed={removed:,} ({pct:.1f}%)")
    return edge_index, removed, pct


print("\nBuilding protocol graphs...")

# 1) Standard
edges_standard = edges_df.copy()
edge_index_standard = build_edge_index(edges_standard)
standard_directed_edges = edge_index_standard.shape[1]

print(f"Standard all edges            | directed edges={standard_directed_edges:,}")

# 2) No test-test
edges_no_test_test = edges_df[
    ~(edges_df["txId1"].isin(test_ids) & edges_df["txId2"].isin(test_ids))
].copy()

# 3) No history-test
edges_no_history_test = edges_df[
    ~(
        (edges_df["txId1"].isin(hist_ids) & edges_df["txId2"].isin(test_ids)) |
        (edges_df["txId1"].isin(test_ids) & edges_df["txId2"].isin(hist_ids))
    )
].copy()

# 4) Fully isolated test
edges_fully_isolated = edges_df[
    ~(edges_df["txId1"].isin(test_ids) | edges_df["txId2"].isin(test_ids))
].copy()

edge_index_no_test_test, removed_ntt, pct_ntt = edge_stats(
    "No test-test", edges_no_test_test, standard_directed_edges
)

edge_index_no_history_test, removed_nht, pct_nht = edge_stats(
    "No history-test", edges_no_history_test, standard_directed_edges
)

edge_index_fully_isolated, removed_fit, pct_fit = edge_stats(
    "Fully isolated test", edges_fully_isolated, standard_directed_edges
)

# -----------------------------
# Models
# -----------------------------
class MLP_Elliptic(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        return self.net(x)


class GraphSAGE_Elliptic(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3):
        super().__init__()

        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.tab = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, 2),
            nn.Softmax(dim=-1)
        )

        self.clf = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

        self.dropout = dropout

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.gelu(self.norm1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index)
        h_gnn = F.gelu(self.norm2(h))

        h_tab = self.tab(x)

        g = self.gate(torch.cat([h_gnn, h_tab], dim=1))
        h_fused = g[:, 0:1] * h_gnn + g[:, 1:2] * h_tab

        return self.clf(h_fused)

# -----------------------------
# Evaluation helpers
# -----------------------------
def best_threshold_from_val(y_val, p_val):
    precision, recall, thresholds = precision_recall_curve(y_val, p_val)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)

    best_i = np.argmax(f1s[:-1])
    return float(thresholds[best_i]), float(f1s[best_i])


def compute_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)

    return {
        "AUC_ROC": roc_auc_score(y_true, probs),
        "AUC_PR": average_precision_score(y_true, probs),
        "F1": f1_score(y_true, pred),
        "Precision": np.sum((pred == 1) & (y_true == 1)) / max(np.sum(pred == 1), 1),
        "Recall": np.sum((pred == 1) & (y_true == 1)) / max(np.sum(y_true == 1), 1),
        "threshold": threshold
    }

# -----------------------------
# Training functions
# -----------------------------
def get_class_weight_train():
    y_train = y_tensor[train_idx].numpy()
    n_neg = np.sum(y_train == 0)
    n_pos = np.sum(y_train == 1)
    return torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)


def train_mlp(protocol_name="MLP no graph", n_epochs=200, lr=1e-3):
    print("\n" + "="*70)
    print(f"Training: {protocol_name}")
    print("="*70)

    model = MLP_Elliptic(len(feat_cols), hidden_dim=128, dropout=0.3).to(device)

    x_gpu = x_tensor.to(device)
    y_gpu = y_tensor.to(device)
    weights = get_class_weight_train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_ap = -1
    best_state = None
    patience = 20
    bad = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(x_gpu)
        loss = F.cross_entropy(logits[train_idx], y_gpu[train_idx], weight=weights)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        model.eval()
        if epoch % 5 == 0:
            with torch.no_grad():
                probs_val = F.softmax(model(x_gpu)[val_idx], dim=1)[:, 1].cpu().numpy()
                y_val = y_tensor[val_idx].numpy()
                val_ap = average_precision_score(y_val, probs_val)

            if val_ap > best_val_ap:
                best_val_ap = val_ap
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1

            if epoch % 25 == 0:
                print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | val_AP={val_ap:.4f} | best={best_val_ap:.4f}")

            if bad >= patience:
                print(f"Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    with torch.no_grad():
        probs_val = F.softmax(model(x_gpu)[val_idx], dim=1)[:, 1].cpu().numpy()
        probs_test = F.softmax(model(x_gpu)[test_idx], dim=1)[:, 1].cpu().numpy()

    y_val = y_tensor[val_idx].numpy()
    y_test = y_tensor[test_idx].numpy()

    threshold, val_f1 = best_threshold_from_val(y_val, probs_val)
    metrics = compute_metrics(y_test, probs_test, threshold)

    metrics.update({
        "protocol": protocol_name,
        "best_val_AP": best_val_ap,
        "val_F1_at_threshold": val_f1,
        "removed_edges": 0,
        "removed_edges_pct": 100.0
    })

    print_result(metrics)
    return metrics


def train_gnn(edge_index, protocol_name, removed_edges=0, removed_edges_pct=0.0, n_epochs=200, lr=1e-3):
    print("\n" + "="*70)
    print(f"Training: {protocol_name}")
    print("="*70)

    model = GraphSAGE_Elliptic(len(feat_cols), hidden_dim=128, dropout=0.3).to(device)

    x_gpu = x_tensor.to(device)
    y_gpu = y_tensor.to(device)
    ei_gpu = edge_index.to(device)
    weights = get_class_weight_train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_ap = -1
    best_state = None
    patience = 20
    bad = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(x_gpu, ei_gpu)
        loss = F.cross_entropy(logits[train_idx], y_gpu[train_idx], weight=weights)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                probs_val = F.softmax(model(x_gpu, ei_gpu)[val_idx], dim=1)[:, 1].cpu().numpy()
                y_val = y_tensor[val_idx].numpy()
                val_ap = average_precision_score(y_val, probs_val)

            if val_ap > best_val_ap:
                best_val_ap = val_ap
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1

            if epoch % 25 == 0:
                print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | val_AP={val_ap:.4f} | best={best_val_ap:.4f}")

            if bad >= patience:
                print(f"Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    with torch.no_grad():
        probs_val = F.softmax(model(x_gpu, ei_gpu)[val_idx], dim=1)[:, 1].cpu().numpy()
        probs_test = F.softmax(model(x_gpu, ei_gpu)[test_idx], dim=1)[:, 1].cpu().numpy()

    y_val = y_tensor[val_idx].numpy()
    y_test = y_tensor[test_idx].numpy()

    threshold, val_f1 = best_threshold_from_val(y_val, probs_val)
    metrics = compute_metrics(y_test, probs_test, threshold)

    metrics.update({
        "protocol": protocol_name,
        "best_val_AP": best_val_ap,
        "val_F1_at_threshold": val_f1,
        "removed_edges": removed_edges,
        "removed_edges_pct": removed_edges_pct
    })

    print_result(metrics)
    return metrics


def print_result(r):
    print(f"\n✅ {r['protocol']}")
    print(f"AUC-ROC   = {r['AUC_ROC']:.4f}")
    print(f"AUC-PR    = {r['AUC_PR']:.4f}")
    print(f"F1        = {r['F1']:.4f}")
    print(f"Precision = {r['Precision']:.4f}")
    print(f"Recall    = {r['Recall']:.4f}")
    print(f"Threshold = {r['threshold']:.4f}")
    print(f"Val AP    = {r['best_val_AP']:.4f}")

# -----------------------------
# Run experiments
# -----------------------------
results = []

results.append(train_mlp("MLP baseline - no graph", n_epochs=200))
torch.cuda.empty_cache()

results.append(train_gnn(edge_index_standard, "GNN Standard - all edges", 0, 0.0, n_epochs=200))
torch.cuda.empty_cache()

results.append(train_gnn(edge_index_no_test_test, "GNN No test-test edges", removed_ntt, pct_ntt, n_epochs=200))
torch.cuda.empty_cache()

results.append(train_gnn(edge_index_no_history_test, "GNN No history-test edges", removed_nht, pct_nht, n_epochs=200))
torch.cuda.empty_cache()

results.append(train_gnn(edge_index_fully_isolated, "GNN Fully isolated test", removed_fit, pct_fit, n_epochs=200))
torch.cuda.empty_cache()

# -----------------------------
# Final table
# -----------------------------
df_results = pd.DataFrame(results)

cols = [
    "protocol", "AUC_ROC", "AUC_PR", "F1", "Precision", "Recall",
    "threshold", "best_val_AP", "removed_edges", "removed_edges_pct"
]

df_results = df_results[cols]
df_results.to_csv(f"{OUT_DIR}/elliptic_external_validation_corrected.csv", index=False)

print("\n" + "="*90)
print("FINAL RESULTS — Elliptic++ External Validation")
print("="*90)
print(df_results.to_string(index=False))

# -----------------------------
# Comparison with BitFraud
# -----------------------------
standard_ap = df_results[df_results["protocol"] == "GNN Standard - all edges"]["AUC_PR"].iloc[0]
isolated_ap = df_results[df_results["protocol"] == "GNN Fully isolated test"]["AUC_PR"].iloc[0]

elliptic_drop = standard_ap - isolated_ap
elliptic_rel_drop = 100 * elliptic_drop / standard_ap if standard_ap > 0 else 0

comparison = pd.DataFrame({
    "dataset": ["BitFraud", "Elliptic++"],
    "standard_AUC_PR": [0.8372, standard_ap],
    "fully_isolated_AUC_PR": [0.0581, isolated_ap],
    "delta_AUC_PR": [0.8372 - 0.0581, elliptic_drop],
    "relative_drop_pct": [84.8, elliptic_rel_drop]
})

comparison.to_csv(f"{OUT_DIR}/comparison_bitfraud_elliptic_corrected.csv", index=False)

print("\n" + "="*90)
print("BITFRAUD vs ELLIPTIC++ — FAIR COMPARISON")
print("="*90)
print(comparison.to_string(index=False))

print(f"\n✅ Saved to: {OUT_DIR}")

Mounted at /content/drive
Device: cuda
Loading Elliptic++ data...
Labeled: 46,564
Fraud rate: 9.76%
Features: 103

Temporal split:
Train: 26,381 | fraud=2,871 | rate=10.88%
Val  : 3,513 | fraud=591 | rate=16.82%
Test : 16,670 | fraud=1,083 | rate=6.50%

Building protocol graphs...
Standard all edges            | directed edges=73,248
No test-test                   | directed edges=45,796 | removed=27,452 (37.5%)
No history-test                | directed edges=73,248 | removed=0 (0.0%)
Fully isolated test            | directed edges=45,796 | removed=27,452 (37.5%)

Training: MLP baseline - no graph
Epoch 025 | loss=0.2445 | val_AP=0.8012 | best=0.8012
Epoch 050 | loss=0.1838 | val_AP=0.8486 | best=0.8486
Epoch 075 | loss=0.1566 | val_AP=0.8601 | best=0.8640
Epoch 100 | loss=0.1351 | val_AP=0.8421 | best=0.8640
Epoch 125 | loss=0.1208 | val_AP=0.8108 | best=0.8640
Epoch 150 | loss=0.1122 | val_AP=0.7951 | best=0.8640
Early stop at epoch 160

✅ MLP baseline - no graph
AUC-ROC   = 0.8928
A